# Text+Tabular BERT Logistic Fusion Transfer Classifier

Train and evaluate a frozen-BERT + tabular fusion model with a logistic regression head:
- train on Canada, test on Canada and US
- train on US, test on US and Canada

Tabular features are transfer-safe intro-time features from `bills.csv`.

## 0. Setup

In [ ]:
import json
import re
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
NORM = DATA / 'normalized'
CA_TXT = DATA / 'canada_bill_text'
US_TXT = DATA / 'us_bill_text'

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 256
CHUNK_STRIDE = 64
MAX_CHUNKS = 2
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-4
SEED = 42
TRAIN_FRAC = 0.7
VAL_FRAC = 0.1
TARGET_COL = 'passed'

NUMERIC_COLS = [
    'year',
    'title_word_count',
    'description_word_count',
    'month_introduced',
]
CATEGORICAL_COLS = [
    'bill_type',
    'chamber',
]

CAT_EMBED_DIM = 16
NUM_PROJ_DIM = 32

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'ROOT: {ROOT}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'DEVICE: {DEVICE}')

## 1. Load Bills and Full Text

In [ ]:
bills_path = NORM / 'bills.csv'
if not bills_path.exists():
    print('bills.csv not found - running normalize.py ...')
    result = subprocess.run(['python', str(ROOT / 'scripts' / 'normalize.py')], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('normalize.py failed')

bills = pd.read_csv(bills_path, low_memory=False)
for col in NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL, 'source', 'session', 'bill_number', 'introduced_date']:
    if col not in bills.columns:
        bills[col] = np.nan

bills[TARGET_COL] = pd.to_numeric(bills[TARGET_COL], errors='coerce').fillna(0).astype(int)
bills['introduced_date'] = pd.to_datetime(bills.get('introduced_date'), errors='coerce')
bills['_row_id'] = np.arange(len(bills), dtype=int)

def _ca_xml_text(el) -> str:
    if el.tag in {'Identification', 'Label', 'MarginalNote'}:
        return ''
    if el.tag == 'AmendedText' and el.get('{http://www.w3.org/XML/1998/namespace}lang') == 'fr':
        return ''
    parts = []
    if el.tag == 'Text' and el.text:
        parts.append(el.text.strip())
    for child in el:
        parts.append(_ca_xml_text(child))
        if child.tail:
            parts.append(child.tail.strip())
    return ' '.join(p for p in parts if p)

def load_canada_texts(ca_txt_dir: Path) -> dict:
    texts = {}
    for session_dir in sorted(ca_txt_dir.iterdir()):
        if not session_dir.is_dir():
            continue
        session = session_dir.name
        for txt_file in session_dir.glob('*_english.txt'):
            bill_num = txt_file.name.replace('.pdf_english.txt', '').replace('_english.txt', '')
            texts[f'{session}/{bill_num}'] = txt_file.read_text(encoding='utf-8', errors='replace')
        for xml_file in session_dir.glob('*.xml'):
            key = f'{session}/{xml_file.stem}'
            if key in texts:
                continue
            try:
                root = ET.parse(xml_file).getroot()
                raw = _ca_xml_text(root)
                if raw:
                    texts[key] = raw
            except Exception:
                pass
    return texts

_SKIP_TAGS = {'metadata', 'dublinCore', 'form', 'distribution-code', 'congress', 'session', 'current-chamber', 'action', 'action-date', 'action-desc', 'legis-type', 'enum', 'external-xref', 'quote'}
_PREFIX_TO_TYPE = {'HB': 'hr', 'SB': 's', 'HR': 'hres', 'SR': 'sres', 'HJR': 'hjres', 'SJR': 'sjres', 'HCR': 'hconres', 'SCR': 'sconres'}
_TYPE_TO_PREFIX = {v: k for k, v in _PREFIX_TO_TYPE.items()}

def _xml_text(el) -> str:
    if el.tag in _SKIP_TAGS:
        return ''
    parts = [el.text or '']
    for child in el:
        parts.append(_xml_text(child))
        parts.append(child.tail or '')
    return ' '.join(p.strip() for p in parts if p.strip())

def load_us_texts(us_txt_dir: Path) -> dict:
    texts = {}
    for zip_path in sorted(us_txt_dir.glob('*.zip')):
        parts = zip_path.stem.split('_')
        congress = parts[0]
        xml_type = parts[-1]
        legiscan_prefix = _TYPE_TO_PREFIX.get(xml_type)
        if legiscan_prefix is None:
            continue
        try:
            with zipfile.ZipFile(zip_path) as zf:
                for name in zf.namelist():
                    if not name.endswith('.xml'):
                        continue
                    match = re.match(rf'BILLS-{congress}{xml_type}(\d+)', name, re.IGNORECASE)
                    if not match:
                        continue
                    key = f'{congress}/{legiscan_prefix}{int(match.group(1))}'
                    try:
                        root = ET.fromstring(zf.read(name))
                        body = root.find('./legis-body')
                        if body is not None:
                            texts[key] = _xml_text(body)
                    except Exception:
                        pass
        except Exception:
            pass
    return texts

ca_texts = load_canada_texts(CA_TXT)
us_texts = load_us_texts(US_TXT)

bills['full_text'] = ''
mask_ca = bills['source'] == 'canada'
mask_us = bills['source'] == 'us'
bills.loc[mask_ca, 'full_text'] = bills[mask_ca].apply(lambda r: ca_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)
bills.loc[mask_us, 'full_text'] = bills[mask_us].apply(lambda r: us_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)

def pick_first_present(df: pd.DataFrame, choices: list[str], default: str = '') -> pd.Series:
    out = pd.Series([default] * len(df), index=df.index, dtype='object')
    for col in choices:
        if col in df.columns:
            vals = df[col].fillna('').astype(str).str.strip()
            out = out.where(out.str.len() > 0, vals)
    return out

title_text = pick_first_present(bills, ['title'])
desc_text = pick_first_present(bills, ['long_description', 'description'])
full_text = bills['full_text'].fillna('').astype(str)
bills['model_text'] = (title_text + '\n' + desc_text + '\n' + full_text).str.replace(r'\s+', ' ', regex=True).str.strip()

print(f'Loaded {len(bills):,} bills')
print('Text coverage by source (model_text non-empty):')
print(bills.groupby('source')['model_text'].apply(lambda s: (s.str.len() > 0).mean()).round(3))

## 2. Country Splits and Metadata State

In [ ]:
@dataclass
class CountrySplit:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame

def make_country_split(df: pd.DataFrame, source: str, train_frac: float = TRAIN_FRAC, val_frac: float = VAL_FRAC) -> CountrySplit:
    country_df = df[(df['source'] == source) & (df['model_text'].str.len() > 0)].copy()
    country_df = country_df.sort_values(['introduced_date', '_row_id']).reset_index(drop=True)
    if len(country_df) < 3:
        raise ValueError(f'Not enough rows for {source} split')

    n_total = len(country_df)
    n_train = max(1, int(n_total * train_frac))
    n_val = max(1, int(n_total * val_frac))
    if n_train + n_val >= n_total:
        n_train = max(1, n_total - 2)
        n_val = 1

    train_df = country_df.iloc[:n_train].copy()
    val_df = country_df.iloc[n_train:n_train + n_val].copy()
    test_df = country_df.iloc[n_train + n_val:].copy()
    if len(test_df) == 0:
        test_df = country_df.iloc[-1:].copy()

    return CountrySplit(train_df=train_df, val_df=val_df, test_df=test_df)

def fit_feature_state(train_df: pd.DataFrame) -> dict:
    num_df = train_df[NUMERIC_COLS].apply(pd.to_numeric, errors='coerce')
    num_means = num_df.mean(numeric_only=True)
    num_stds = num_df.std(numeric_only=True).replace(0, 1).fillna(1)

    cat_maps = {}
    cat_cardinalities = []
    for col in CATEGORICAL_COLS:
        values = ['<PAD>', '<UNK>'] + sorted(v for v in train_df[col].fillna('').astype(str).unique().tolist() if v)
        mapping = {v: i for i, v in enumerate(values)}
        cat_maps[col] = mapping
        cat_cardinalities.append(len(values))

    return {
        'num_means': num_means,
        'num_stds': num_stds,
        'cat_maps': cat_maps,
        'cat_cardinalities': cat_cardinalities,
    }

def transform_metadata(df: pd.DataFrame, feature_state: dict) -> tuple[np.ndarray, np.ndarray]:
    cat_arrays = []
    for col in CATEGORICAL_COLS:
        mapping = feature_state['cat_maps'][col]
        cat_arrays.append(df[col].fillna('').astype(str).map(lambda x, m=mapping: m.get(x, 1)).to_numpy())
    cat_x = np.stack(cat_arrays, axis=1).astype('int64')

    num_df = df[NUMERIC_COLS].apply(pd.to_numeric, errors='coerce')
    num_x = ((num_df.fillna(feature_state['num_means']) - feature_state['num_means']) / feature_state['num_stds']).astype('float32').to_numpy()
    return cat_x, num_x

country_splits = {
    'canada': make_country_split(bills, 'canada'),
    'us': make_country_split(bills, 'us'),
}
for country, split in country_splits.items():
    print(f"{country.upper()}: train={len(split.train_df):,}, val={len(split.val_df):,}, test={len(split.test_df):,}")

## 3. Tokenization and Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_text(text: str, tokenizer, max_length: int = MAX_LENGTH, stride: int = CHUNK_STRIDE):
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']

class TextTabularDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_state: dict):
        self.df = df.reset_index(drop=True)
        self.texts = self.df['model_text'].fillna('').astype(str).tolist()
        self.cat_x, self.num_x = transform_metadata(self.df, feature_state)
        self.labels = self.df[TARGET_COL].to_numpy(dtype='float32')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'categorical': torch.tensor(self.cat_x[idx], dtype=torch.long),
            'numerical': torch.tensor(self.num_x[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def collate_batch(batch):
    texts = [item['text'] for item in batch]
    cats = torch.stack([item['categorical'] for item in batch])
    nums = torch.stack([item['numerical'] for item in batch])
    labels = torch.stack([item['label'] for item in batch])

    chunked_ids = []
    chunked_masks = []
    max_chunks = 1
    for text in texts:
        input_ids, attention_mask = chunk_text(text, tokenizer)
        input_ids = input_ids[:MAX_CHUNKS]
        attention_mask = attention_mask[:MAX_CHUNKS]
        chunked_ids.append(input_ids)
        chunked_masks.append(attention_mask)
        max_chunks = max(max_chunks, input_ids.size(0))

    seq_len = chunked_ids[0].size(-1)
    padded_ids = []
    padded_masks = []
    for input_ids, attention_mask in zip(chunked_ids, chunked_masks):
        if input_ids.size(0) < max_chunks:
            pad_shape = (max_chunks - input_ids.size(0), seq_len)
            id_pad = torch.zeros(pad_shape, dtype=torch.long)
            mask_pad = torch.zeros(pad_shape, dtype=torch.long)
            input_ids = torch.cat([input_ids, id_pad], dim=0)
            attention_mask = torch.cat([attention_mask, mask_pad], dim=0)
        padded_ids.append(input_ids)
        padded_masks.append(attention_mask)

    return {
        'input_ids': torch.stack(padded_ids),
        'attention_mask': torch.stack(padded_masks),
        'categorical': cats,
        'numerical': nums,
        'labels': labels,
    }

## 4. Logistic Fusion Model

In [ ]:
class TextTabularLogisticFusionClassifier(nn.Module):
    def __init__(self, model_name: str, cat_cardinalities: list[int], num_numeric: int, cat_embed_dim: int = CAT_EMBED_DIM, num_proj_dim: int = NUM_PROJ_DIM, freeze_bert: bool = True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.text_dim = getattr(self.bert.config, 'hidden_size', 768)
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.cat_embeddings = nn.ModuleList([nn.Embedding(cardinality, cat_embed_dim) for cardinality in cat_cardinalities])
        self.num_projection = nn.Linear(num_numeric, num_proj_dim)

        self.cat_block_dim = len(cat_cardinalities) * cat_embed_dim
        self.num_block_dim = num_proj_dim
        fusion_dim = self.text_dim + self.cat_block_dim + self.num_block_dim

        # Logistic regression fusion head: single linear layer (no MLP hidden layers).
        self.logistic_head = nn.Linear(fusion_dim, 1)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        batch_size, num_chunks, seq_len = input_ids.shape
        flat_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        outputs = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        return cls_embeddings.mean(dim=1)

    def encode_tabular(self, categorical: torch.Tensor, numerical: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        cat_parts = []
        for idx, embedding in enumerate(self.cat_embeddings):
            cat_parts.append(embedding(categorical[:, idx]))
        cat_vec = torch.cat(cat_parts, dim=1) if cat_parts else categorical.new_zeros((categorical.size(0), 0), dtype=torch.float32)
        num_vec = self.num_projection(numerical)
        return cat_vec, num_vec

    def forward(self, input_ids, attention_mask, categorical, numerical):
        text_vec = self.encode_text(input_ids, attention_mask)
        cat_vec, num_vec = self.encode_tabular(categorical, numerical)
        fused = torch.cat([text_vec, cat_vec, num_vec], dim=1)
        return self.logistic_head(fused).squeeze(-1)

print('TextTabularLogisticFusionClassifier defined.')

## 5. Training, Transfer, and Logistic Feature Importance

In [ ]:
def make_pos_weight(labels: np.ndarray) -> torch.Tensor:
    positives = max(float(labels.sum()), 1.0)
    negatives = max(float(len(labels) - labels.sum()), 1.0)
    return torch.tensor([negatives / positives], dtype=torch.float32)

def make_loader(df: pd.DataFrame, feature_state: dict, shuffle: bool = False):
    ds = TextTabularDataset(df, feature_state)
    return torch.utils.data.DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_batch,
        pin_memory=(DEVICE.type == 'cuda'),
    )

def train_one_epoch(model, loader, optimizer, criterion, device, grad_accum_steps=1, use_amp=False, scaler=None, show_progress=False, progress_desc='Train'):
    model.train()
    total_loss = 0.0
    total_examples = 0
    optimizer.zero_grad(set_to_none=True)
    iterator = tqdm(loader, desc=progress_desc, leave=False, dynamic_ncols=True) if show_progress else loader
    for step, batch in enumerate(iterator, start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        categorical = batch['categorical'].to(device)
        numerical = batch['numerical'].to(device)
        labels = batch['labels'].to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids, attention_mask, categorical, numerical)
            raw_loss = criterion(logits, labels)
            loss = raw_loss / grad_accum_steps

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % grad_accum_steps == 0 or step == len(loader):
            if use_amp and scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item() * labels.size(0)
        total_examples += labels.size(0)
        if show_progress and hasattr(iterator, 'set_postfix'):
            iterator.set_postfix(loss=f'{raw_loss.item():.4f}')

    return total_loss / max(total_examples, 1)

def predict_loader(model, loader, device):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                batch['categorical'].to(device),
                batch['numerical'].to(device),
            )
            all_logits.append(logits.detach().cpu())
            all_labels.append(batch['labels'].detach().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    return labels, probs, logits

def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1_vals))])

def compute_metrics(y_true, y_prob, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }

def extract_logistic_importance(model: TextTabularLogisticFusionClassifier, feature_state: dict) -> dict:
    weights = model.logistic_head.weight.detach().cpu().numpy()[0]
    bias = float(model.logistic_head.bias.detach().cpu().numpy()[0])

    text_end = model.text_dim
    cat_end = text_end + model.cat_block_dim
    num_end = cat_end + model.num_block_dim

    text_coef = weights[:text_end]
    cat_coef = weights[text_end:cat_end]
    num_proj_coef = weights[cat_end:num_end]

    top_text_idx = np.argsort(np.abs(text_coef))[::-1][:20]
    text_table = pd.DataFrame({
        'bert_dim': top_text_idx,
        'coef': text_coef[top_text_idx],
        'abs_coef': np.abs(text_coef[top_text_idx]),
    })

    cat_rows = []
    start = 0
    for col_idx, col in enumerate(CATEGORICAL_COLS):
        end = start + CAT_EMBED_DIM
        col_coef = cat_coef[start:end]
        emb = model.cat_embeddings[col_idx].weight.detach().cpu().numpy()
        reverse_map = {idx: val for val, idx in feature_state['cat_maps'][col].items()}

        for idx in range(emb.shape[0]):
            category = reverse_map.get(idx, f'idx_{idx}')
            contribution = float(np.dot(col_coef, emb[idx]))
            cat_rows.append({
                'feature': col,
                'category': category,
                'logit_contribution': contribution,
                'abs_contribution': abs(contribution),
            })
        start = end

    cat_table = pd.DataFrame(cat_rows).sort_values('abs_contribution', ascending=False).reset_index(drop=True)

    proj_w = model.num_projection.weight.detach().cpu().numpy()
    std_coef = np.matmul(num_proj_coef, proj_w)
    raw_coef = std_coef / np.clip(feature_state['num_stds'].to_numpy(dtype='float32'), 1e-6, None)
    numeric_table = pd.DataFrame({
        'feature': NUMERIC_COLS,
        'coef_on_standardized_input': std_coef,
        'coef_approx_raw_scale': raw_coef,
        'abs_coef': np.abs(raw_coef),
    }).sort_values('abs_coef', ascending=False).reset_index(drop=True)

    return {
        'bias': bias,
        'text_coefficients': text_table,
        'categorical_contributions': cat_table,
        'numeric_coefficients': numeric_table,
    }

def train_country_model(train_df: pd.DataFrame, val_df: pd.DataFrame, train_source: str):
    feature_state = fit_feature_state(train_df)
    train_loader = make_loader(train_df, feature_state, shuffle=True)
    val_loader = make_loader(val_df, feature_state, shuffle=False)

    model = TextTabularLogisticFusionClassifier(
        model_name=MODEL_NAME,
        cat_cardinalities=feature_state['cat_cardinalities'],
        num_numeric=len(NUMERIC_COLS),
        freeze_bert=True,
    ).to(DEVICE)

    pos_weight = make_pos_weight(train_df[TARGET_COL].to_numpy(dtype='float32')).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    history = []
    grad_accum_steps = max(1, 8 // max(1, BATCH_SIZE))
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            DEVICE,
            grad_accum_steps=grad_accum_steps,
            use_amp=(DEVICE.type == 'cuda'),
            scaler=scaler,
            show_progress=True,
            progress_desc=f'{train_source.title()} train {epoch}/{EPOCHS}',
        )

        val_labels, val_probs, _ = predict_loader(model, val_loader, DEVICE)
        val_thr = best_f1_threshold(val_labels, val_probs)
        val_metrics = compute_metrics(val_labels, val_probs, threshold=val_thr)

        history.append({'epoch': epoch, 'train_loss': train_loss, **{k: v for k, v in val_metrics.items() if k != 'confusion_matrix'}})
        print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_pr_auc={val_metrics['pr_auc']:.4f} | val_f1={val_metrics['f1']:.4f}")

    history_df = pd.DataFrame(history)
    final_val_labels, final_val_probs, _ = predict_loader(model, val_loader, DEVICE)
    threshold = best_f1_threshold(final_val_labels, final_val_probs)

    importance = extract_logistic_importance(model, feature_state)
    return model, feature_state, threshold, history_df, importance

def run_logistic_transfer_experiment(train_source: str, country_splits: dict[str, CountrySplit]) -> dict:
    train_split = country_splits[train_source]
    other_source = 'us' if train_source == 'canada' else 'canada'
    other_split = country_splits[other_source]

    model, feature_state, threshold, history_df, importance = train_country_model(train_split.train_df, train_split.val_df, train_source)

    in_loader = make_loader(train_split.test_df, feature_state, shuffle=False)
    out_loader = make_loader(other_split.test_df, feature_state, shuffle=False)

    in_y, in_prob, _ = predict_loader(model, in_loader, DEVICE)
    out_y, out_prob, _ = predict_loader(model, out_loader, DEVICE)

    in_metrics = compute_metrics(in_y, in_prob, threshold=threshold)
    out_metrics = compute_metrics(out_y, out_prob, threshold=threshold)

    rows = [
        {'train_country': train_source, 'test_country': train_source, 'setting': 'in_country_test', **in_metrics},
        {'train_country': train_source, 'test_country': other_source, 'setting': 'transfer_test', **out_metrics},
    ]

    return {
        'train_country': train_source,
        'other_country': other_source,
        'threshold': threshold,
        'history': history_df,
        'report': pd.DataFrame(rows),
        'importance': importance,
    }

print('Training, transfer, and importance helpers defined.')

## 6. Canada-Only Training Run

In [ ]:
print('Running Canada-only logistic fusion transfer experiment...')
canada_run = run_logistic_transfer_experiment('canada', country_splits)

print('\nCanada training history:')
display(canada_run['history'])

print('\nCanada transfer report:')
display(canada_run['report'])

print('\nCanada model: top numeric coefficients')
display(canada_run['importance']['numeric_coefficients'].head(10))

print('\nCanada model: top categorical contributions')
display(canada_run['importance']['categorical_contributions'].head(20))

## 7. US-Only Training Run and Final Report

In [ ]:
print('Running US-only logistic fusion transfer experiment...')
us_run = run_logistic_transfer_experiment('us', country_splits)

print('\nUS training history:')
display(us_run['history'])

print('\nUS transfer report:')
display(us_run['report'])

print('\nUS model: top numeric coefficients')
display(us_run['importance']['numeric_coefficients'].head(10))

print('\nUS model: top categorical contributions')
display(us_run['importance']['categorical_contributions'].head(20))

final_report = pd.concat([canada_run['report'], us_run['report']], ignore_index=True)
final_report = final_report[[
    'train_country',
    'test_country',
    'setting',
    'threshold',
    'accuracy',
    'balanced_accuracy',
    'pr_auc',
    'roc_auc',
    'f1',
    'precision',
    'recall',
]]

print('\nCombined logistic-fusion transfer report:')
display(final_report)

report_path = NORM / 'text_tabular_logistic_transfer_report.json'
with open(report_path, 'w', encoding='utf-8') as fp:
    json.dump(final_report.to_dict(orient='records'), fp, indent=2)
print(f'Saved transfer report to {report_path}')

def _importance_payload(run: dict) -> dict:
    return {
        'train_country': run['train_country'],
        'threshold': run['threshold'],
        'bias': run['importance']['bias'],
        'numeric_coefficients': run['importance']['numeric_coefficients'].to_dict(orient='records'),
        'categorical_contributions': run['importance']['categorical_contributions'].to_dict(orient='records'),
        'text_coefficients': run['importance']['text_coefficients'].to_dict(orient='records'),
    }

canada_imp_path = NORM / 'text_tabular_logistic_feature_importance_canada.json'
us_imp_path = NORM / 'text_tabular_logistic_feature_importance_us.json'
with open(canada_imp_path, 'w', encoding='utf-8') as fp:
    json.dump(_importance_payload(canada_run), fp, indent=2)
with open(us_imp_path, 'w', encoding='utf-8') as fp:
    json.dump(_importance_payload(us_run), fp, indent=2)

print(f'Saved Canada importance to {canada_imp_path}')
print(f'Saved US importance to {us_imp_path}')